# BaseLine Logit, self training only model seed 2023, How well does self-training model label using unseen text file? unlabel_percentage = 0.3

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.semi_supervised import SelfTrainingClassifier
import pandas as pd
import joblib

# https://www.codecademy.com/resources/docs/sklearn/self-training
# https://www.altexsoft.com/blog/semi-supervised-learning/
# https://scikit-learn.org/stable/modules/semi_supervised.html
# This self-training implementation is based on Yarowsky’s [1] algorithm.
# https://towardsdatascience.com/self-training-classifier-how-to-make-any-algorithm-behave-like-a-semi-supervised-one-2958e7b54ab7/

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")


print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})

vectorizer = TfidfVectorizer()

review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# Array representing the labels. Unlabeled samples should have the label -1.
review_y = review_data["recommend"].fillna(-1)


# print(vectorizer.get_feature_names_out())  # Shows vocabulary
review_X_train, review_X_test, review_y_train, review_y_test = train_test_split(review_X, review_y, test_size=0.2, random_state=2023)

print(f"count for missing data {len(review_y_train[review_y_train==-1])}")
print(f"count for labeled data {len(review_y_train[review_y_train!=-1])}")
print(f"count for missing data_test {len(review_y_test [review_y_test ==-1])}")
print(f"count for labeled data_Test {len(review_y_test [review_y_test !=-1])}")


logit_review_model = LogisticRegression()





self_training_logit_model = SelfTrainingClassifier(logit_review_model,verbose=True)

self_training_logit_model.fit(review_X_train, review_y_train)


print(sum(review_y_pred==-1))
print(sum(review_y_test==-1))

# create a mask
mask = review_y_test != -1 
review_X_test =   review_X_test[mask]
review_y_test =  review_y_test[mask]


review_y_pred = self_training_logit_model.predict(review_X_test)


# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test , review_y_pred)
precision = precision_score(review_y_test , review_y_pred ,average='macro')
recall = recall_score(review_y_test, review_y_pred ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"Accuracy: {accuracy:.4f}")


joblib.dump(self_training_logit_model, "base_logistic_model.pkl")
print("\nModel saved successfully.")

1
count for missing data 20073
count for labeled data 46744
count for missing data_test 4983
count for labeled data_Test 11722
End of iteration 1, added 14534 new labels.
End of iteration 2, added 803 new labels.
End of iteration 3, added 143 new labels.
End of iteration 4, added 77 new labels.
End of iteration 5, added 10 new labels.
End of iteration 6, added 13 new labels.
End of iteration 7, added 1 new labels.
End of iteration 8, added 7 new labels.
End of iteration 9, added 23 new labels.
End of iteration 10, added 16 new labels.
0
4983

Evaluation Results on Test Data:
Accuracy:  0.8544616959563215
Precision: 0.8474009780276295
Recall:    0.8120987990824307
Accuracy: 0.8545

Model saved successfully.


This tells us how the model is able to predict labels it has not seen before but does not tell us how accurate the model actually impute the data set necessarly


The data for random seed 2025 seem to very specifically yeild 0 as an accuracy result seems like I randomly pick the worse training data for it. Since logistic is fast to train I decided to make some random seed and average it

# BaseLine Logit, self training only model seed 2023, How well does self-training model impute the entire file. Not implmented yet

Metrics entire dataset,  All prediction/total lableled data

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.semi_supervised import SelfTrainingClassifier
import pandas as pd
import joblib

# https://www.codecademy.com/resources/docs/sklearn/self-training
# https://www.altexsoft.com/blog/semi-supervised-learning/
# https://scikit-learn.org/stable/modules/semi_supervised.html
# This self-training implementation is based on Yarowsky’s [1] algorithm.
# https://towardsdatascience.com/self-training-classifier-how-to-make-any-algorithm-behave-like-a-semi-supervised-one-2958e7b54ab7/

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")


print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})

vectorizer = TfidfVectorizer()

review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# Array representing the labels. Unlabeled samples should have the label -1.
review_y = review_data["recommend"].fillna(-1)


# print(vectorizer.get_feature_names_out())  # Shows vocabulary
# review_X_train, review_X_test, review_y_train, review_y_test = train_test_split(review_X, review_y, test_size=0.2, random_state=2023)

# print(f"count for missing data {len(review_y_train[review_y_train==-1])}")
# print(f"count for labeled data {len(review_y_train[review_y_train!=-1])}")
# print(f"count for missing data_test {len(review_y_test [review_y_test ==-1])}")
# print(f"count for labeled data_Test {len(review_y_test [review_y_test !=-1])}")


logit_review_model = LogisticRegression()



def train_sk_single_model(dataset,model)
# dataset should be a tuple here 
    X, y = dataset
    model.fit(train_x, train_y,)
    return model


def train_



self_training_logit_model = SelfTrainingClassifier(logit_review_model,verbose=True)

# fit using the entire data set
self_training_logit_model.fit(review_X , review_y)



#-------making the test data file from full data


review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")


print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})



review_X_test= vectorizer.transform(review_data_test ["review_comment"])  # Convert text to TF-IDF features
review_y_test = review_data_test["recommend"]


# review_X_test is the same as review_X so we are just retriveving the predicted label

review_y_pred = self_training_logit_model.predict(review_X_test)


# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, review_y_pred)
precision = precision_score(review_y_test, review_y_pred ,average='macro')
recall = recall_score(review_y_test, review_y_pred ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"Accuracy: {accuracy:.4f}")

1
End of iteration 1, added 18336 new labels.
End of iteration 2, added 981 new labels.
End of iteration 3, added 209 new labels.
End of iteration 4, added 49 new labels.
End of iteration 5, added 20 new labels.
End of iteration 6, added 22 new labels.
End of iteration 7, added 8 new labels.
End of iteration 8, added 53 new labels.
End of iteration 9, added 28 new labels.
End of iteration 10, added 5 new labels.
1

Evaluation Results on Test Data:
Accuracy:  0.8732310050046694
Precision: 0.8695981871592748
Recall:    0.8361131558801127
Accuracy: 0.8732


# Evaluatiing on only unlabeled data

Unlabeled prediction/unlableled data

Using the unlabeled data X as input to predict how many are labeled correctly

In [8]:

review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")


print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})


review_X_test= vectorizer.transform(review_data_test ["review_comment"])  # Convert text to TF-IDF features
review_y_test = review_data_test["recommend"]


# only the unlabled data
mask = review_y == -1 
review_X_test =   review_X_test[mask]
review_y_test =  review_y_test[mask]


review_y_pred = self_training_logit_model.predict(review_X_test)


# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, review_y_pred)
precision = precision_score(review_y_test, review_y_pred ,average='macro')
recall = recall_score(review_y_test, review_y_pred ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"Accuracy: {accuracy:.4f}")

1

Evaluation Results on Test Data:
Accuracy:  0.8614303959131545
Precision: 0.8557344357789043
Recall:    0.8229421484510033
Accuracy: 0.8614


  # Implementation with random f to new data

In [33]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.semi_supervised import SelfTrainingClassifier
import pandas as pd
import joblib

# https://www.codecademy.com/resources/docs/sklearn/self-training

# https://scikit-learn.org/stable/modules/semi_supervised.html
# This self-training implementation is based on Yarowsky’s [1] algorithm.
# https://towardsdatascience.com/self-training-classifier-how-to-make-any-algorithm-behave-like-a-semi-supervised-one-2958e7b54ab7/

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")


print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})

vectorizer = TfidfVectorizer()

review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# Array representing the labels. Unlabeled samples should have the label -1.
review_y = review_data["recommend"].fillna(-1)


# print(vectorizer.get_feature_names_out())  # Shows vocabulary
review_X_train, review_X_test, review_y_train, review_y_test = train_test_split(review_X, review_y, test_size=0.2, random_state=2023)


random_forest_model = RandomForestClassifier(n_estimators=30, random_state=2023)



self_training_logit_model = SelfTrainingClassifier(random_forest_model,verbose=True)

self_training_logit_model.fit(review_X_train, review_y_train)

mask = review_y_test != -1 
review_X_test =   review_X_test[mask]
review_y_test =  review_y_test[mask]

review_y_pred = self_training_logit_model.predict(review_X_test)

# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, review_y_pred )
precision = precision_score(review_y_test, review_y_pred ,average='macro')
recall = recall_score(review_y_test, review_y_pred ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"Accuracy: {accuracy:.4f}")


joblib.dump(self_training_logit_model, "selftrain_noboot_rf_model.pkl")
print("\nModel saved successfully.")


with open('evaluation_results_selftrain_noboot_rf_model.txt', 'w') as f:
    f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")


1
End of iteration 1, added 11663 new labels.
End of iteration 2, added 1743 new labels.
End of iteration 3, added 931 new labels.
End of iteration 4, added 601 new labels.
End of iteration 5, added 451 new labels.
End of iteration 6, added 283 new labels.
End of iteration 7, added 256 new labels.
End of iteration 8, added 208 new labels.
End of iteration 9, added 169 new labels.
End of iteration 10, added 162 new labels.

Evaluation Results on Test Data:
Accuracy:  0.8244326906671217
Precision: 0.8159755537925456
Recall:    0.7696756895909413
Accuracy: 0.8244

Model saved successfully.


# Random forest evluation 

Metrics entire dataset,  All prediction/total lableled data

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.semi_supervised import SelfTrainingClassifier
import pandas as pd
import joblib

# https://www.codecademy.com/resources/docs/sklearn/self-training
# https://www.altexsoft.com/blog/semi-supervised-learning/
# https://scikit-learn.org/stable/modules/semi_supervised.html
# This self-training implementation is based on Yarowsky’s [1] algorithm.
# https://towardsdatascience.com/self-training-classifier-how-to-make-any-algorithm-behave-like-a-semi-supervised-one-2958e7b54ab7/

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")


print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})

vectorizer = TfidfVectorizer()

review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# Array representing the labels. Unlabeled samples should have the label -1.
review_y = review_data["recommend"].fillna(-1)


# print(vectorizer.get_feature_names_out())  # Shows vocabulary
# review_X_train, review_X_test, review_y_train, review_y_test = train_test_split(review_X, review_y, test_size=0.2, random_state=2023)

# print(f"count for missing data {len(review_y_train[review_y_train==-1])}")
# print(f"count for labeled data {len(review_y_train[review_y_train!=-1])}")
# print(f"count for missing data_test {len(review_y_test [review_y_test ==-1])}")
# print(f"count for labeled data_Test {len(review_y_test [review_y_test !=-1])}")


random_forest_model = RandomForestClassifier(n_estimators=30, random_state=2023)





self_training_logit_model = SelfTrainingClassifier(random_forest_model ,verbose=True)

self_training_logit_model.fit(review_X , review_y)



#-------making the test data file from full data


review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")


print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})

vectorizer = TfidfVectorizer()

review_X_test= vectorizer.fit_transform(review_data_test ["review_comment"])  # Convert text to TF-IDF features
review_y_test = review_data_test["recommend"]




review_y_pred = self_training_logit_model.predict(review_X_test)


# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, review_y_pred)
precision = precision_score(review_y_test, review_y_pred ,average='macro')
recall = recall_score(review_y_test, review_y_pred ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"Accuracy: {accuracy:.4f}")

1
count for missing data 20073
count for labeled data 46744
count for missing data_test 0
count for labeled data_Test 83522
End of iteration 1, added 14767 new labels.
End of iteration 2, added 2144 new labels.
End of iteration 3, added 1023 new labels.
End of iteration 4, added 704 new labels.
End of iteration 5, added 507 new labels.
End of iteration 6, added 359 new labels.
End of iteration 7, added 331 new labels.
End of iteration 8, added 276 new labels.
End of iteration 9, added 246 new labels.
End of iteration 10, added 202 new labels.
1

Evaluation Results on Test Data:
Accuracy:  0.9427456239074735
Precision: 0.9435867727562016
Recall:    0.9252796810941468
Accuracy: 0.9427


In [52]:

with open('evaluation_results_selftrain_noboot__fulldta_rf_model.txt', 'w') as f:
    f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")


# only unlabeld

Unlabeled prediction/unlableled data

In [ ]:

review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")



print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})


review_X_test= vectorizer.fit_transform(review_data_test ["review_comment"])  # Convert text to TF-IDF features
review_y_test = review_data_test["recommend"]


# only the unlabled data
mask = review_y == -1 
review_X_test =   review_X_test[mask]
review_y_test =  review_y_test[mask]

review_y_pred = self_training_logit_model.predict(review_X_test)


# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, review_y_pred)
precision = precision_score(review_y_test, review_y_pred ,average='macro')
recall = recall_score(review_y_test, review_y_pred ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
print(f"Accuracy: {accuracy:.4f}")

1

Evaluation Results on Test Data:
Accuracy:  0.838521711366539
Precision: 0.8337905730866886
Recall:    0.7895399069999248
Accuracy: 0.8385


In [2]:

def stat_bootstrap_df(data, num_samples=1000):
    """
    Perform bootstrapping with replacement using pandas to estimate the sampling distribution of a statistic.
    
    Parameters:
    - data: A pandas Series or DataFrame.
    - num_samples: The number of bootstrap samples to generate.

    Returns:
    - bootstrap_samples: A pandas Series containing bootstrap sample statistics.
    """
    # Ensure data is a pandas Series
    if isinstance(data, pd.DataFrame):
        data = data.squeeze()  # Converts DataFrame to Series if needed
    
    bootstrap_samples = []

    for _ in range(num_samples):
        # Generate a bootstrap sample by randomly sampling with replacement
        sample = data.sample(n=len(data), replace=True)
        # Calculate the statistic (e.g., mean)
        bootstrap_samples.append(sample)

    return pd.Series(bootstrap_samples)

# bert self trianing 

In [5]:
import torch

if torch.backends.mps.is_available():
    print("MPS is available.")
else:
    print("MPS is NOT available.")

# Optional: Also check if it's built (important for some setups)
if torch.backends.mps.is_built():
    print("MPS backend is built into PyTorch.")
else:
    print("MPS backend is NOT built into PyTorch.")

MPS is available.
MPS backend is built into PyTorch.


In [ ]:
def get_predictions(model, data, batch_size=64):
    """ Get predictions on data for a classification model m returning predictions and true labels"""
    model.to("cpu")
    model.eval()
 
    all_predictions = []
    all_confidences = []
    
    if len(data['input_ids']) % batch_size != 0:
        num_batches = len(data['input_ids']) // batch_size + 1
    else:
        num_batches = len(data['input_ids']) // batch_size

    for i in tqdm(range(num_batches),desc= "Prediction Progress"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(data['input_ids'])) # prevent going out of bount input_ids has same size as attention mask should
        input_ids_batch = torch.tensor(data['input_ids'][start_idx:end_idx])
        attention_mask_batch = torch.tensor(data['attention_mask'][start_idx:end_idx])
        with torch.no_grad():  # Disable gradient computation for inference
            predictions = model(input_ids_batch, attention_mask=attention_mask_batch)
            logits = predictions.logits
            predictions = torch.argmax(logits, dim=-1) # make sure it is a ser
            probs = torch.nn.functional.softmax(logits, dim=-1) # return a proabiblity distribution
            max_probs, _ = torch.max(probs, dim=-1)
            
        all_confidences.append(max_probs)  # this tells us how confident we are with an value
        all_predictions.append(predictions)
        # print(predictions)
        # print("--------")
    # print(all_predictions)
    
    
    #Concatenate all batch predictions into one tensor since we currently have a list of lists
    all_predictions = torch.cat(all_predictions, dim=0)
    all_confidences = torch.cat(all_confidences, dim=0)
    
    return all_predictions , all_confidences


def self_training_loop(model, train_dataset, tokenizer, confidence_threshold=0.9, num_epochs=1, mode='unlabeled',threshold_mode = 'fix'):
    
    # still need to better develop still bad code since exit really randomly depend on the confidence_threshold which is not good for big data waiting for the final data point to set in
    
    
    data_collator = DataCollatorWithPadding(tokenizer) 
    training_args = TrainingArguments(
        num_train_epochs=num_epochs,      # Number of training epochs
    )

    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,    # Training dataset   
        data_collator=data_collator,
    )
    
    #https://github.com/tqdm/tqdm#usage
    unlabeled_mask =  train_dataset["label"] == -1
    total_unlabeled = sum(label == -1 for label in train_dataset["label"])
    pbar = tqdm(total=total_unlabeled, desc="Unlabeled Samples Remaining")

    labeled_train_dataset = train_dataset.filter(lambda x: x["label"] != -1)
    trainer.train_dataset = labeled_train_dataset  # Update trainer with labeled dataset
    print(f"data: { labeled_train_dataset =}")
    # https://peps.python.org/pep-0289/
    print(sum(label == -1 for label in labeled_train_dataset["label"]))
    print("hello")
    trainer.train()

    while -1 in train_dataset["label"]:
        # trainer.train()
        predictions, confidences = get_predictions(model, train_dataset)
        
        if threshold_mode == 'fix':
            threshold_value = confidence_threshold
        elif threshold_mode == 'quantile':
            # this mode with too higih quantile does have converge issue
            threshold_value = torch.quantile(confidences, confidence_threshold).item()
        else:
            print("threshold mode not specified")

        print(confidences)
        print(predictions)
        print(len(train_dataset))
        print("------------------------")
        if mode == 'unlabeled':
        # we might want to make distinction here is the train_dataset label only or etc
        #  It allows you to apply a processing function to each example in a dataset, independently or in batches. This function can even create new rows and columns.
        # https://huggingface.co/docs/datasets/en/process
            condition_met_indices = [] # this is just here to keep track of the pesudo labeled added per iteration if needed
            # example here is each row
            def update_labels(example, idx):
                # Update labels for unlabeled data only, if confidence is above the threshold
                if example['label'] == -1 and confidences[idx] >= threshold_value:
                    example['label'] = predictions[idx]
                    condition_met_indices.append(idx)
                return example

            # Use .map() to apply the updates to all examples where the condition is met
            train_dataset = train_dataset.map(update_labels, with_indices=True)
            print(f"Indices where the condition was met: {condition_met_indices}")
                  

        # untested yet !!!!!!!
        elif mode == "all":
            def update_labels_all(example, idx):
                if confidences[idx] >= threshold_value:
                    example['label'] = predictions[idx]
                    condition_met_indices.append(idx)
                return example

            train_dataset = train_dataset.map(update_labels_all, with_indices=True)
            print(f"Indices where the condition was met: {condition_met_indices}")
                    
        # Update trainer with the latest training data
        pbar.update(len(condition_met_indices))
        trainer.train_dataset = train_dataset
        trainer.train()
            
        if -1 not in train_dataset["label"]:
          print("All labels are filled. Stopping training.")
          break
        
    return model, train_dataset

# testing for 5000 sample

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
from trl import IterativeSFTTrainer
from tqdm import tqdm


# We are still in datafre

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv",nrows=5000)

print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})



# vectorizer = TfidfVectorizer()

# review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# # Array representing the labels. Unlabeled samples should have the label -1.

review_data["recommend"] = review_data["recommend"].fillna(-1)


print("wow")
print(review_data[review_data["recommend"]==-1])


# for some reason the tyepe matter https://discuss.huggingface.co/t/valueerror-target-size-torch-size-8-must-be-the-same-as-input-size-torch-size-8-8/12133/9
train_df = pd.DataFrame({"text": review_data["review_comment"], "label": review_data["recommend"].astype(int)})



# Now we are working with huggingface daset 

train_dataset = Dataset.from_pandas(train_df)

print(sum(label == -1 for label in train_dataset["label"]))
print(train_dataset["label"])
unlabeled_mask =  train_dataset["label"] == -1
print(unlabeled_mask)
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Bert model accepts max 512
# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)

#------------------------------------


#------------------------------------

# In Hugging Face models like BERT, the features come from the tokenized input data that is fed into the model.
# the map applie the tokenize function to each batch of the function 
train_dataset = train_dataset.map(tokenize, batched=True)

# print(train_dataset)
# Load model
# define model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


model, train_dataset = self_training_loop(model, train_dataset,tokenizer)

# Save the trained model and tokenizer
model.save_pretrained("self_trained_bert_model")
tokenizer.save_pretrained("self_trained_tokenizer")

# Save the final labeled dataset
train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)


#---------Evaluate-----------



train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)
review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv",nrows=5000)



print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)



review_y_test = review_data_test["recommend"]

train_dataset_frame =  train_dataset.to_pandas()
# only the unlabled data
mask = review_data["recommend"] == -1 
review_y_test =  review_y_test[mask]

final_data_set = train_dataset_frame[mask]
print(review_y_test)
print(final_data_set)

# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, final_data_set["label"] )
precision = precision_score(review_y_test, final_data_set["label"]  ,average='macro')
recall = recall_score(review_y_test,final_data_set["label"]  ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")

with open('evaluation_results_selftrain_noboot_bert_models.txt', 'w') as f:
    f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")



Evlautation

Unlabeled prediction/unlableled data test on full data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
from trl import IterativeSFTTrainer
from tqdm import tqdm




# We are still in datafre

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")

print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})



# vectorizer = TfidfVectorizer()

# review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# # Array representing the labels. Unlabeled samples should have the label -1.

review_data["recommend"] = review_data["recommend"].fillna(-1)


# print("wow")
# print(review_data[review_data["recommend"]==-1])


# for some reason the tyepe matter https://discuss.huggingface.co/t/valueerror-target-size-torch-size-8-must-be-the-same-as-input-size-torch-size-8-8/12133/9
train_df = pd.DataFrame({"text": review_data["review_comment"], "label": review_data["recommend"].astype(int)})



# Now we are working with huggingface daset 

train_dataset = Dataset.from_pandas(train_df)

print(sum(label == -1 for label in train_dataset["label"]))
print(train_dataset["label"])
unlabeled_mask =  train_dataset["label"] == -1
# print(unlabeled_mask)
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Bert model accepts max 512
# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)

#------------------------------------


#------------------------------------

# In Hugging Face models like BERT, the features come from the tokenized input data that is fed into the model.
# the map applie the tokenize function to each batch of the function 
train_dataset = train_dataset.map(tokenize, batched=True)

# print(train_dataset)
# Load model
# define model



# print (f"we are using {device}")

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
device = torch.device("cpu")
model.to(device)


model, train_dataset = self_training_loop(model, train_dataset, threshold_mode = 'fix')

# Save the trained model and tokenizer
model.save_pretrained("self_trained_bert_final_model")
tokenizer.save_pretrained("self_trained_final_tokenizer")

# Save the final labeled dataset
train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)


#---------Evaluate-----------



train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)
review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")



print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)



review_y_test = review_data_test["recommend"]

train_dataset_frame =  train_dataset.to_pandas()
# only the unlabled data
mask = review_data["recommend"] == -1 
review_y_test =  review_y_test[mask]

final_data_set = train_dataset_frame[mask]
# print(review_y_test)
# print(final_data_set)

# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, final_data_set["label"] )
precision = precision_score(review_y_test, final_data_set["label"]  ,average='macro')
recall = recall_score(review_y_test,final_data_set["label"]  ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")

with open('evaluation_results_selftrain_noboot_full_bert_models.txt', 'w') as f:
    f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")



1
25056
[-1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1, 1, 1, 0, 1, 1, -1, 1, 0, 1, -1, 1, -1, 1, -1, 0, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1, -1, 1, -1, 1, -1, 0, -1, 1, -1, 1, -1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 0, 1, 1, -1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, -1, -1, -1, 1, -1, 1, 1, 1, 1, -1, 1, -1, 0, -1, 1, -1, 0, 1, 0, -1, -1, -1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, -1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1, -1, -1, 1, 1, 1, -1, -1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, -1, 1, 1, 1, 1, -1, 0, 1, 1, 1, 1, 1, -1, 1, 1, 0, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1, -1, 1, -1, 1, 1, 1, -1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, -1, -1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, -1, -1, -1, -1, -1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, -1, -1, 1, 1, -1, 1, -1, -1, 1, 1, 1, 1, -1

Map:   0%|          | 0/83522 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unlabeled Samples Remaining:   0%|          | 0/25056 [00:00<?, ?it/s]

Filter:   0%|          | 0/83522 [00:00<?, ? examples/s]

data:  labeled_train_dataset =Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 58466
})
0
hello


...

/opt/anaconda3/envs/cosc410_test/lib/python3.10/site-packages/torch/utils/data/dataloader.py:685: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.129100
1000,0.152000
1500,0.167700
2000,0.214200
2500,0.176700
3000,0.220200
3500,0.216500
4000,0.179500
4500,0.220100
5000,0.215300


Unlabeled Samples Remaining: 100%|██████████| 25056/25056 [236:18:38<00:00, 33.95s/it] 

All labels are filled. Stopping training.


1

Evaluation Results on Test Data:
Accuracy:  0.9040948275862069
Precision: 0.8941426768586276
Recall:    0.8868404755714969
